# 07 — Fleet-wide Volt-VAr slope analysis (2025, month by month)

**Method** (validated in `05_exploratory_data_analysis.ipynb`, Section 12), per site-month:

1. Polarity-corrected net meter reading (`ami_meter_residential` × `circuit_polarity`).
2. Night-time baseline, **per circuit and per month**: `P_est = base_P − P_net`, `Q_est = base_Q − Q_net`.
3. **Signed net-export gate**: keep intervals with `P_est ≥ 20%` of rating (|P| would also admit household import).
4. OLS of `Q_est` (% of rating) on `V` over the assessable window [240, 253) V.

**Sites**: `n_load_phases == 1` and `n_pv_phases == 1`, excluding `flex_export_detected`.

**How it runs**: all filtering and fitting happens inside DuckDB (`lib/ami_fleet_vvar.py`), so only
one row per site-month reaches Python. Each month is written to its own Parquet file; if the run
stops, just re-run the cell — finished months are skipped.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Put the repo root on sys.path (same as the other notebooks).
_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_conformance_plot as ConformancePlot

## 1. Settings

In [ ]:
MONTHS = [f"2025-{m:02d}" for m in range(1, 13)]

# Method settings (defaults = the values validated in notebook 05)
NIGHT_HOURS = (0, 4)             # AEST hours for the monthly night-time baseline
P_MIN_FRAC = Fleet.P_MIN_FRAC    # signed gate: P_est >= 20% of rating
V_LO, V_HI = Fleet.V_LO, Fleet.V_HI
FIT_SETTINGS = dict(night_hours=NIGHT_HOURS, p_min_frac=P_MIN_FRAC, v_lo=V_LO, v_hi=V_HI)

# Fit-quality thresholds -- applied AFTER the run, so they can change without re-running
MIN_N_WINDOW = 20       # min. 5-min intervals in the assessable window per site-month (~8 h)
MIN_V_SPREAD = 5.0       # min. voltage range (V) covered by those intervals

# Pass/fail -- still to decide; leave as None to skip the pass/fail cells
SLOPE_TOLERANCE_PCT = 20

# DuckDB resources (spills to %LOCALAPPDATA%/ciccada/duckdb_tmp, outside OneDrive)
DUCKDB_MEMORY, DUCKDB_THREADS = "8GB", 4

# Outputs live next to the data store (local disk, not OneDrive)
OUT_DIR = Fleet.store_dir() / "vvar_fleet" / "2025_baseline_signed_gate"
print(f"Results folder: {OUT_DIR}")

## 2. Site eligibility

In [ ]:
site_meta = pd.read_parquet(Fleet.Config.store_path("ami_site_metadata_residential"))
eligibility = Fleet.build_eligibility(site_meta)
sites = eligibility.loc[eligibility.eligibility == "eligible"]

display(eligibility.eligibility.value_counts().rename("n_sites").to_frame())
print(f"{len(sites):,} eligible sites (1 load phase, 1 PV phase, no flex export, export limit OK).")

## 3. Run, month by month (resumable)

Each month takes one pass over that month's Parquet partition. Safe to interrupt and re-run:
months already in `OUT_DIR` are skipped (`overwrite=True` forces a redo). Progress is also
appended to `run_log.csv` in each results folder.

In [ ]:
# Blind analysis: net meter + monthly baseline + signed gate
log_blind = Fleet.run_months(
    MONTHS, OUT_DIR / "blind", sites, source="meter_baseline",
    memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, **FIT_SETTINGS,
)
display(log_blind)

In [ ]:
# Ground truth: the separate PV circuit, same window and gate (no baseline needed)
log_truth = Fleet.run_months(
    MONTHS, OUT_DIR / "truth", sites, source="ground_truth",
    memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, **FIT_SETTINGS,
)
display(log_truth)

## 4. Load results and apply fit-quality thresholds

In [ ]:
blind = Fleet.add_fit_status(Fleet.load_results(OUT_DIR / "blind"), min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)
truth = Fleet.add_fit_status(Fleet.load_results(OUT_DIR / "truth"), min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)

# Sanity check: one row per site-month (a single-phase site should have one circuit)
for name, df in [("blind", blind), ("truth", truth)]:
    n_dup = df.duplicated(["site_id", "year", "month"]).sum()
    assert n_dup == 0, f"{name}: {n_dup} duplicated site-months -- a site has >1 circuit"

# Site-months by fit status, per month
display(pd.crosstab([blind.year, blind.month], blind.fit_status, margins=True))

## 5. Validation against ground truth

Where both the blind fit and the ground-truth fit are `ok` for the same site-month: how close is
the blind slope to the real inverter's?

In [ ]:
compare = Fleet.compare_to_truth(blind, truth)
err = compare.slope_err.abs()
print(f"{len(compare):,} site-months with both fits ok")
print(f"|slope error| (%/V): median {err.median():.3f}, p75 {err.quantile(.75):.3f}, "
      f"p90 {err.quantile(.9):.3f}, share < 0.5 %/V: {(err < 0.5).mean():.1%}")

Fleet.plot_blind_vs_truth(compare)
plt.show()

In [ ]:
# Worst disagreements -- candidates for a site-level flag (see the Section-12 tail sites)
display(compare.sort_values("slope_err", key=abs, ascending=False).head(15).round(3))

## 6. Fleet results

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), dpi=130, sharey=True)
Fleet.plot_monthly_slopes(blind, "Blind (net meter + monthly baseline + signed gate)", ax=axes[0])
Fleet.plot_monthly_slopes(truth, "Ground truth (separate PV circuit)", ax=axes[1])
fig.tight_layout()
plt.show()

In [ ]:
# Level/offset of each fitted line vs the required line at V3, V4 and their midpoint
# (reuses the notebook-05 helper). Offsets from the blind fit carry the known positive
# baseline bias (household load Q), so read them with that caveat.
blind_ok = ConformancePlot.add_offset_pct_columns(blind.loc[blind.fit_status == "ok"])
truth_ok = ConformancePlot.add_offset_pct_columns(truth.loc[truth.fit_status == "ok"])

summary = pd.concat([
    df.groupby(["year", "month"]).agg(n_sites=("site_id", "nunique"), median_slope=("slope", "median"),
                                      median_r2=("r_squared", "median"), median_offset_mid=("offset_pct_mid", "median"))
      .assign(source=name)
    for name, df in [("blind", blind_ok), ("truth", truth_ok)]
]).reset_index().pivot(index=["year", "month"], columns="source")
display(summary.round(3))

In [ ]:
# Pass/fail on slope (runs once SLOPE_TOLERANCE_PCT is set in Section 1)
if SLOPE_TOLERANCE_PCT is None:
    print("SLOPE_TOLERANCE_PCT not set yet -- skipping pass/fail.")
else:
    blind_pf = ConformancePlot.add_slope_pass_fail(blind_ok, SLOPE_TOLERANCE_PCT)
    truth_pf = ConformancePlot.add_slope_pass_fail(truth_ok, SLOPE_TOLERANCE_PCT)

    # Monthly pass rate
    pass_rate = pd.DataFrame({
        "blind_%": blind_pf.groupby(["year", "month"]).passed.mean() * 100,
        "truth_%": truth_pf.groupby(["year", "month"]).passed.mean() * 100,
    })
    display(pass_rate.round(1))

    # Agreement: does the blind pass/fail match the ground-truth pass/fail for the same site-month?
    keys = ["site_id", "year", "month"]
    both = blind_pf[keys + ["passed"]].merge(truth_pf[keys + ["passed"]], on=keys, suffixes=("_blind", "_truth"))
    display(pd.crosstab(both.passed_blind, both.passed_truth, rownames=["blind"], colnames=["truth"]))
    print(f"Agreement: {(both.passed_blind == both.passed_truth).mean():.1%} of {len(both):,} site-months")

In [ ]:
# Pass/fail on slope (runs once SLOPE_TOLERANCE_PCT is set in Section 1)
POSITIVE_IS_FAIL = True   # positive class = non-conformant (fails the slope test); False -> positive = passes

def classification_metrics(pred: pd.Series, actual: pd.Series) -> dict:
    """Confusion counts + accuracy/precision/recall/specificity/F1 (NaN where undefined)."""
    tp = int((pred & actual).sum());   fp = int((pred & ~actual).sum())
    fn = int((~pred & actual).sum());  tn = int((~pred & ~actual).sum())
    div = lambda a, b: a / b if b else float("nan")
    precision, recall = div(tp, tp + fp), div(tp, tp + fn)
    return {"n": tp + fp + fn + tn, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": div(tp + tn, tp + fp + fn + tn), "precision": precision, "recall": recall,
            "specificity": div(tn, tn + fp), "f1": div(2 * precision * recall, precision + recall)}

if SLOPE_TOLERANCE_PCT is None:
    print("SLOPE_TOLERANCE_PCT not set yet -- skipping pass/fail.")
else:
    blind_pf = ConformancePlot.add_slope_pass_fail(blind_ok, SLOPE_TOLERANCE_PCT)
    truth_pf = ConformancePlot.add_slope_pass_fail(truth_ok, SLOPE_TOLERANCE_PCT)

    # Monthly pass rate
    pass_rate = pd.DataFrame({
        "blind_%": blind_pf.groupby(["year", "month"]).passed.mean() * 100,
        "truth_%": truth_pf.groupby(["year", "month"]).passed.mean() * 100,
    })
    display(pass_rate.round(1))

    # Same site-month under both sources (ground truth = the reference label)
    keys = ["site_id", "year", "month"]
    cols = keys + ["passed", "slope", "pct_error", "r_squared", "n_window"]
    both = blind_pf[cols].merge(truth_pf[cols], on=keys, suffixes=("_blind", "_truth"))
    display(pd.crosstab(both.passed_blind, both.passed_truth, rownames=["blind"], colnames=["truth"]))

    # Positive class: fail (non-conformant) by default
    both["pred_pos"] = ~both.passed_blind if POSITIVE_IS_FAIL else both.passed_blind
    both["true_pos"] = ~both.passed_truth if POSITIVE_IS_FAIL else both.passed_truth
    positive_label = "fail (non-conformant)" if POSITIVE_IS_FAIL else "pass (conformant)"

    # Metrics: all months together, then month by month
    overall = pd.DataFrame([classification_metrics(both.pred_pos, both.true_pos)], index=["all months"])
    monthly = pd.DataFrame({f"{y}-{m:02d}": classification_metrics(g.pred_pos, g.true_pos)
                            for (y, m), g in both.groupby(["year", "month"])}).T
    print(f"Positive class = {positive_label}; reference = ground truth; tolerance ±{SLOPE_TOLERANCE_PCT}%")
    display(pd.concat([overall, monthly]).round(3))

    # Export the misclassified site-months (FP = blind flags positive, truth doesn't; FN = the reverse)
    both["outcome"] = np.select(
        [both.pred_pos & ~both.true_pos, ~both.pred_pos & both.true_pos],
        ["false_positive", "false_negative"], default="correct")
    misclassified = both.loc[both.outcome != "correct"].sort_values(["outcome", "site_id", "year", "month"])
    csv_path = OUT_DIR / f"pass_fail_misclassified_tol{SLOPE_TOLERANCE_PCT}.csv"
    misclassified.drop(columns=["pred_pos", "true_pos"]).to_csv(csv_path, index=False)
    print(f"{(misclassified.outcome == 'false_positive').sum():,} FP and "
          f"{(misclassified.outcome == 'false_negative').sum():,} FN site-months "
          f"({misclassified.site_id.nunique():,} distinct sites) -> {csv_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Conformance rate -- BLIND results only. Denominator = assessable site-months (fit_status == "ok").
# ---------------------------------------------------------------------------
assert SLOPE_TOLERANCE_PCT is not None, "Set SLOPE_TOLERANCE_PCT in Section 1 and run the pass/fail cell first."

MIN_GROUP_SITES = 30        # hide groups smaller than this in the displayed tables (noisy rates)
SITE_PASS_SHARE = 0.5       # site-level: conformant if it passes in >= this share of its assessable months
INSTALL_GROUPING = "year"   # "year" or "month" (month-year of pv_install_date)
STANDARD_MANDATORY = pd.Timestamp("2021-12-18")  # AS/NZS 4777.2:2020 mandatory for new installs -- check this date before publishing

# Blind pass/fail + site attributes
meta = site_meta[["site_id", "manufacturer", "dnsp_name", "pv_install_date"]].rename(
    columns={"manufacturer": "oem", "dnsp_name": "dnsp", "pv_install_date": "install_date"})
pf = blind_pf[["site_id", "year", "month", "passed"]].merge(meta, on="site_id", how="left")
pf[["oem", "dnsp"]] = pf[["oem", "dnsp"]].fillna("Unknown")
pf["install"] = (pf.install_date.dt.year if INSTALL_GROUPING == "year"
                 else pf.install_date.dt.to_period("M").astype(str))
pf["install_era"] = np.where(pf.install_date >= STANDARD_MANDATORY,
                             "installed after 2020 standard mandatory", "installed before")

# Funnel per month: eligible -> with data -> assessable -> conformant
funnel = blind.groupby(["year", "month"]).agg(
    eligible_sites=("site_id", "nunique"),
    with_data=("fit_status", lambda s: (s != "no_data").sum()),
    assessable=("fit_status", lambda s: (s == "ok").sum()),
)
funnel["conformant"] = pf.groupby(["year", "month"]).passed.sum()
funnel["conformant_%_of_assessable"] = (100 * funnel.conformant / funnel.assessable).round(1)
display(funnel)

In [ ]:
def rate_table(df: pd.DataFrame, by, flag: str = "passed", min_sites: int = MIN_GROUP_SITES) -> pd.DataFrame:
    """Conformance rate per group: n sites, n rows (site-months or sites), % conformant. Small groups hidden."""
    g = df.groupby(by)
    out = pd.DataFrame({"n_sites": g.site_id.nunique(), "n_rows": g.size(),
                        "conformant_%": (100 * g[flag].mean()).round(1)})
    hidden = (out.n_sites < min_sites).sum()
    if hidden:
        print(f"({hidden} group(s) with < {min_sites} sites hidden)")
    return out.loc[out.n_sites >= min_sites].sort_values("n_sites", ascending=False)

# Whole of 2025, site-month basis
all_rate = 100 * pf.passed.mean()
print(f"2025 overall: {all_rate:.1f}% of {len(pf):,} assessable site-months "
      f"({pf.site_id.nunique():,} sites) conformant at ±{SLOPE_TOLERANCE_PCT}% slope tolerance")

# Site-level: one verdict per site across its assessable months
sites_pf = (pf.groupby("site_id")
              .agg(n_months=("passed", "size"), share_pass=("passed", "mean"),
                   oem=("oem", "first"), dnsp=("dnsp", "first"),
                   install=("install", "first"), install_era=("install_era", "first"))
              .reset_index())
sites_pf["conformant"] = sites_pf.share_pass >= SITE_PASS_SHARE
print(f"Site level: {100 * sites_pf.conformant.mean():.1f}% of {len(sites_pf):,} sites conformant "
      f"(pass in >= {SITE_PASS_SHARE:.0%} of their assessable months)")
display(sites_pf.n_months.describe().round(1).to_frame("assessable months per site"))

In [ ]:
# Each breakdown at two levels: site-months (all of 2025) and sites (one verdict per site)
for label, col in [("OEM", "oem"), ("DNSP", "dnsp"),
                   (f"Install {INSTALL_GROUPING}", "install"), ("Install era", "install_era")]:
    print(f"\n=== Conformance by {label} ===")
    table = rate_table(pf, col).join(
        rate_table(sites_pf, col, flag="conformant", min_sites=0)[["conformant_%"]]
            .rename(columns={"conformant_%": "site_level_conformant_%"}),
        how="left")
    display(table.rename(columns={"n_rows": "n_site_months", "conformant_%": "site_month_conformant_%"}))

## 7. Sensitivity (one month)

Re-runs one month with alternative settings into separate folders (each run is ~a minute).
Compared against the main ground-truth run, so this isolates the effect on the blind method.

In [ ]:
SENS_MONTH = "2025-01"
SENSITIVITY = {
    "gate_15pct":   dict(FIT_SETTINGS, p_min_frac=0.15),
    "gate_30pct":   dict(FIT_SETTINGS, p_min_frac=0.30),
    "v_from_242":   dict(FIT_SETTINGS, v_lo=242),
    "night_00_03":  dict(FIT_SETTINGS, night_hours=(0, 3)),
}

rows = []
truth_month = truth.loc[truth.month == int(SENS_MONTH[-2:])]
for name, settings in {"main": FIT_SETTINGS, **SENSITIVITY}.items():
    out = OUT_DIR / "blind" if name == "main" else OUT_DIR / "sensitivity" / name
    Fleet.run_months([SENS_MONTH], out, sites, memory_limit=DUCKDB_MEMORY, threads=DUCKDB_THREADS, **settings)
    res = Fleet.add_fit_status(Fleet.load_results(out), min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)
    res = res.loc[res.month == int(SENS_MONTH[-2:])]
    cmp = Fleet.compare_to_truth(res, truth_month)
    rows.append({"setting": name, "n_ok": (res.fit_status == "ok").sum(),
                 "median_slope": res.loc[res.fit_status == "ok", "slope"].median(),
                 "median_abs_err_vs_truth": cmp.slope_err.abs().median()})
display(pd.DataFrame(rows).round(3))

## 8. Save outputs

In [ ]:
# One tidy table for later notebooks / the paper: every site-month, both sources, with status
results = pd.concat([blind, truth], ignore_index=True)
results.to_parquet(OUT_DIR / "site_month_results.parquet", index=False)
eligibility.to_parquet(OUT_DIR / "site_eligibility.parquet", index=False)
print(f"Saved {len(results):,} rows -> {OUT_DIR / 'site_month_results.parquet'}")
print(f"Saved {len(eligibility):,} rows -> {OUT_DIR / 'site_eligibility.parquet'}")